# Chapter 9: Docker Compose + Dev Containers

**Solve the "Works on My Machine" problem with containerized environments**

This notebook demonstrates:
1. Testing DuckDB in containerized environment
2. Connecting to MinIO (S3-compatible storage)
3. Running Prefect flows
4. Querying from Evidence dashboards

**Note:** Run this notebook inside the Docker container for full functionality.

## Setup

In [ ]:
# Verify installation
import sys
print(f"Python version: {sys.version}")

import duckdb
print(f"DuckDB version: {duckdb.__version__}")

import polars as pl
print(f"Polars version: {pl.__version__}")

import pyarrow as pa
print(f"PyArrow version: {pa.__version__}")

print("\n[OK] All dependencies installed")

## 1. Test DuckDB

Verify DuckDB works in containerized environment.

In [ ]:
# Simple query
result = duckdb.query("""
    SELECT 
        42 as answer,
        'Hello from Docker' as message,
        current_timestamp as ts
""").df()

print(result)

## 2. Generate Sample Data

Create synthetic sales data for testing.

In [ ]:
from pathlib import Path
import random
from datetime import datetime, timedelta

# Create data directories
Path('data/raw').mkdir(parents=True, exist_ok=True)
Path('data/curated').mkdir(parents=True, exist_ok=True)

# Generate sample data
def generate_sales(n=1000):
    """Generate synthetic sales data."""
    sales = []
    start_date = datetime(2024, 1, 1)
    
    for i in range(n):
        sales.append({
            'order_id': i + 1,
            'customer_id': random.randint(1, 100),
            'amount': round(random.uniform(10, 1000), 2),
            'order_date': (start_date + timedelta(days=random.randint(0, 90))).strftime('%Y-%m-%d'),
            'status': random.choice(['pending', 'shipped', 'delivered'])
        })
    
    return pl.DataFrame(sales)

sales_df = generate_sales(1000)
print(f"Generated {len(sales_df):,} sales records\n")
print(sales_df.head())

In [ ]:
# Save to CSV (raw)
sales_df.write_csv('data/raw/sales.csv')
print("[OK] Saved to data/raw/sales.csv")

# Save to Parquet (curated)
sales_df.write_parquet('data/curated/sales.parquet', compression='zstd')
print("[OK] Saved to data/curated/sales.parquet")

## 3. Query with DuckDB

Test DuckDB's ability to query both CSV and Parquet.

In [ ]:
# Query CSV
csv_result = duckdb.query("""
    SELECT 
        status,
        COUNT(*) as order_count,
        SUM(amount) as total_revenue,
        AVG(amount) as avg_order_value
    FROM read_csv('data/raw/sales.csv')
    GROUP BY status
    ORDER BY total_revenue DESC
""").df()

print("Sales by Status (from CSV):")
print(csv_result)

In [ ]:
# Query Parquet (faster)
parquet_result = duckdb.query("""
    SELECT 
        order_date,
        COUNT(*) as orders,
        SUM(amount) as revenue
    FROM read_parquet('data/curated/sales.parquet')
    GROUP BY order_date
    ORDER BY order_date DESC
    LIMIT 10
""").df()

print("\nDaily Sales (from Parquet):")
print(parquet_result)

## 4. MinIO Integration (S3-Compatible Storage)

Test connection to MinIO and upload/query Parquet files.

In [ ]:
# Test MinIO connection
try:
    from minio import Minio
    
    client = Minio(
        'minio:9000',
        access_key='minioadmin',
        secret_key='minioadmin',
        secure=False
    )
    
    # Create bucket if not exists
    if not client.bucket_exists('analytics'):
        client.make_bucket('analytics')
        print("[OK] Created 'analytics' bucket")
    else:
        print("[OK] 'analytics' bucket exists")
    
    # List buckets
    buckets = client.list_buckets()
    print(f"\nAvailable buckets: {[b.name for b in buckets]}")
    
except Exception as e:
    print(f"[ERROR] MinIO connection failed: {e}")
    print("Make sure MinIO is running: docker compose up -d minio")

In [ ]:
# Upload Parquet to MinIO
try:
    client.fput_object(
        'analytics',
        'curated/sales/sales.parquet',
        'data/curated/sales.parquet'
    )
    print("[OK] Uploaded sales.parquet to MinIO")
except Exception as e:
    print(f"[ERROR] Upload failed: {e}")

In [ ]:
# Query MinIO with DuckDB
try:
    # Configure DuckDB for MinIO
    duckdb.execute("""
        INSTALL httpfs;
        LOAD httpfs;
        SET s3_endpoint='minio:9000';
        SET s3_access_key_id='minioadmin';
        SET s3_secret_access_key='minioadmin';
        SET s3_use_ssl=false;
        SET s3_url_style='path';
    """)
    
    # Query from S3
    s3_result = duckdb.query("""
        SELECT 
            COUNT(*) as total_orders,
            SUM(amount) as total_revenue,
            MIN(order_date) as first_order,
            MAX(order_date) as last_order
        FROM read_parquet('s3://analytics/curated/sales/*.parquet')
    """).df()
    
    print("[OK] Queried Parquet from MinIO (S3):")
    print(s3_result)
    
except Exception as e:
    print(f"[ERROR] S3 query failed: {e}")

## 5. Polars Transformations

Test Polars in containerized environment.

In [ ]:
# Load with Polars
df = pl.read_parquet('data/curated/sales.parquet')

# Lazy transformations
result = (
    df.lazy()
    .filter(pl.col('amount') > 100)
    .group_by('status')
    .agg([
        pl.count().alias('high_value_orders'),
        pl.col('amount').sum().alias('total_revenue'),
        pl.col('amount').mean().alias('avg_order_value')
    ])
    .sort('total_revenue', descending=True)
    .collect()
)

print("High-Value Orders (>$100):")
print(result)

## 6. Environment Check

Verify we're running inside Docker container.

In [ ]:
import os

print("Environment Variables:")
print(f"  AWS_ENDPOINT_URL: {os.getenv('AWS_ENDPOINT_URL', 'not set')}")
print(f"  PREFECT_API_URL: {os.getenv('PREFECT_API_URL', 'not set')}")

print(f"\nWorking Directory: {os.getcwd()}")

# Check if running in container
if os.path.exists('/.dockerenv'):
    print("\n[OK] Running inside Docker container")
else:
    print("\n[INFO] Not running in Docker (or Docker detection failed)")

## 7. Service Connectivity Check

Test connections to other services in the stack.

In [ ]:
import requests

services = {
    'Prefect': 'http://prefect:4200/api/health',
    'MinIO': 'http://minio:9000/minio/health/live',
}

print("Service Health Checks:\n")
for name, url in services.items():
    try:
        response = requests.get(url, timeout=5)
        if response.status_code == 200:
            print(f"  ✓ {name}: OK")
        else:
            print(f"  ✗ {name}: HTTP {response.status_code}")
    except Exception as e:
        print(f"  ✗ {name}: {e}")

print("\nAccess services:")
print("  Prefect UI:    http://localhost:4200")
print("  MinIO Console: http://localhost:9001")
print("  Evidence:      http://localhost:3000")

## Summary

You've tested:
- ✅ DuckDB queries on CSV and Parquet
- ✅ Polars transformations
- ✅ MinIO (S3-compatible) storage
- ✅ DuckDB querying from S3
- ✅ Service connectivity

**Next steps:**
1. Run Prefect flows: `python3 etl/flows/ingest.py`
2. Set up Evidence dashboards
3. Add more services to docker-compose.yaml
4. Share with teammates: `git clone && docker compose up`

**Key benefit:** This notebook runs identically on any machine with Docker.